import dependencies

In [ ]:
import pandas as pd
import numpy as np

import data

In [2]:
BASE = "../datasets/P02. MySpotify"

In [3]:
df1 = pd.read_csv(
    f"{BASE}/train_triplets.txt",
    sep="\t", header=None,
    names=["user_id", "song_id", "play_count"]
)

print(df1.shape)
df1.sample(5)

(48373586, 3)


,user_id,song_id,play_count
11249591,0d1ae4b3cbcf9dc794d0e9e293392e0794c43d6c,SOTIYWM12AB0189BBD,1
10152836,f4e9128e234226dfd04e7a2f1a460c48fa76542b,SORNZQX12A58A771FD,1
47014223,2489e0d52d37966670b501dd1a4b50ec9d0bf704,SOWERHT12AB017D08E,3
18021635,e7e257a177338ad801a1e42ef589dbe4d81401e4,SOMXXET12AC4688DD6,1
40974618,4af3ed89b3e4c8c34547c708505f58aa4df3e759,SOHYKBC12A58A7CDD2,1


In [4]:
df2 = pd.read_csv(
    f"{BASE}/p02_unique_tracks.txt",
    sep="<SEP>", engine="python", header=None,
    names=["track_id", "song_id", "artist", "title"]
)

print(df2.shape)
df2.head(5)

(1000000, 4)


,track_id,song_id,artist,title
0,TRMMMYQ128F932D901,SOQMMHC12AB0180CB8,Faster Pussy cat,Silent Night
1,TRMMMKD128F425225D,SOVFVAK12A8C1350D9,Karkkiautomaatti,Tanssi vaan
2,TRMMMRX128F93187D9,SOGTUKN12AB017F4F1,Hudson Mohawke,No One Could Ever
3,TRMMMCH128F425532C,SOBNYVR12A8C13558C,Yerba Brava,Si Vos Querés
4,TRMMMWA128F426B589,SOHSBXH12A8C13B0DF,Der Mystic,Tangle Of Aspens


In [5]:
df3 = pd.read_csv(
    f"{BASE}/p02_msd_tagtraum_cd2.cls",
    sep="\t", comment="#", header=None,
    names=["track_id", "majority_genre", "minority_genre"]
)

print(df3.shape)
df3.sample(5)

(280831, 3)


,track_id,majority_genre,minority_genre
142637,TRMJHKP128F92F7B84,Rock,NaN
115947,TRJXPBD128F93305A6,New Age,NaN
111366,TRJMTOM128F9329309,Jazz,NaN
36750,TRCZMIC128F93363B4,Folk,NaN
176694,TRPOOZW128F425149B,Rock,NaN


In [6]:
vocab = None
mxm_rows = []
with open(f"{BASE}/mxm_dataset_train.txt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("%"):
            vocab = line[1:].split(",")
            continue
        parts = line.split(",")
        mxm_rows.append({
            "track_id": parts[0],
            "mxm_track_id": parts[1],
            "word_counts": ",".join(parts[2:])
        })
df4 = pd.DataFrame(mxm_rows)

print(df4.shape)
df4.head(5)

(210519, 3)


,track_id,mxm_track_id,word_counts
0,TRAAAAV128F421A322,4623710,"1:6,2:4,3:2,4:2,5:5,6:3,7:1,8:1,11:1,12:2,13:3..."
1,TRAAABD128F429CF47,6477168,"1:10,3:17,4:8,5:2,6:2,7:1,8:3,9:2,10:3,11:4,12..."
2,TRAAAED128E0783FAB,2516445,"1:28,2:15,3:2,4:12,5:22,6:2,7:2,8:4,9:2,10:1,1..."
3,TRAAAEF128F4273421,3759847,"1:5,2:4,3:3,4:2,5:1,6:11,9:4,12:9,13:3,15:2,16..."
4,TRAAAEW128F42930C0,3783760,"1:4,4:5,5:7,6:2,7:4,9:1,10:1,11:9,15:1,17:2,20..."


df1 — Play counts - датасет количества прослушанных треков
(48,373,586 × 3) — user_id, song_id, play_count

df2 — Unique tracks - датасет треков
(1,000,000 × 4) — track_id, song_id, artist, title

df3 — Genre annotations - датасет жанров по трекам (популярный, второй по популярности)
(280,831 × 3) — track_id, majority_genre, minority_genre

df4 — Lyrics bag-of-words (musiXmatch) - датасет с контектом песен (мешок слов)
(210,519 × 3) — track_id, mxm_track_id, word_counts

EDA

In [7]:
bins = [0, 1, 2, 3, 5, 10, 20, 50, 100, df1["play_count"].max()]
labels = ["1", "2", "3", "4-5", "6-10", "11-20", "21-50", "51-100", "100+"]
df1["play_count_bucket"] = pd.cut(df1["play_count"], bins=bins, labels=labels)

bucket_counts = df1["play_count_bucket"].value_counts().reindex(labels)
print(bucket_counts)
print()
print((bucket_counts / len(df1) * 100).round(2), "% of all records")

play_count_bucket
1         28755966
2          7336825
3          3214271
4-5        4056080
6-10       2966862
11-20      1381230
21-50       560498
51-100       83043
100+         18811
Name: count, dtype: int64

play_count_bucket
1         59.45
2         15.17
3          6.64
4-5        8.38
6-10       6.13
11-20      2.86
21-50      1.16
51-100     0.17
100+       0.04
Name: count, dtype: float64 % of all records


In [8]:
dup_count = df2["song_id"].duplicated().sum()
print(f"Дублирующихся song_id в df2: {dup_count}")
print(f"Всего строк в df2: {df2.shape[0]}")
print(f"Уникальных song_id: {df2['song_id'].nunique()}")

Дублирующихся song_id в df2: 944
Всего строк в df2: 1000000
Уникальных song_id: 999056


In [9]:
dup_song_ids = df2[df2["song_id"].duplicated(keep=False)]["song_id"].unique()
sample_dups = df2[df2["song_id"].isin(dup_song_ids[:10])].sort_values("song_id")
display(sample_dups)

,track_id,song_id,artist,title
814,TRMMTTM128F92FD3CF,SOAOOXO12AF72A787F,Foo Fighters,Still
443859,TRAPJFL128F42AA7B4,SOAOOXO12AF72A787F,Foo Fighters,Still
969384,TRYRGOU128F42A4AC2,SOCAXWE12A8C13CEC0,System of a Down,This Cocaine Makes Me Feel Like I'm On This Song
487,TRMMQFP128F92F0413,SOCAXWE12A8C13CEC0,System of a Down,This Cocaine Makes Me Feel Like I'm On This Song
501266,TRPMQVR128F4290E34,SOCXXKA12A58A8059A,Dilated Peoples,Mr. Slow Flow (Remix)
6063,TRMCWHJ128F4227D18,SOCXXKA12A58A8059A,Dilated Peoples,Mr Slow Flow featuring PMD (Remix)
766935,TRSKFTA128E0780E33,SOFWNQG12AF72A0FC2,Blink-182,Boring
353,TRMMBDE128F932F640,SOFWNQG12AF72A0FC2,Blink-182,Boring
32760,TRMDYQQ128F428080C,SOJHKXX12A8C139798,Incubus,A Crow Left Of The Murder
5354,TRMHUEV12903CB0E5C,SOJHKXX12A8C139798,Incubus,A Crow Left Of The Murder


In [10]:
df2 = df2.drop_duplicates(subset="song_id", keep="first").reset_index(drop=True)

print("После дедупликации:", df2.shape)
print("Уникальных song_id:", df2["song_id"].nunique())

После дедупликации: (999056, 4)
Уникальных song_id: 999056


In [11]:
print(df3.shape)
df3.isna().sum() # 68% пропусков для minority_genre

(280831, 3)


track_id               0
majority_genre         0
minority_genre    191401
dtype: int64

In [12]:
print(df3["majority_genre"].value_counts())

majority_genre
Rock          113150
Electronic     30922
Pop            27756
Jazz           17541
Rap            13272
RnB            13208
Metal          11222
Country        10695
Reggae          8910
Blues           8125
Latin           6444
Folk            6377
Punk            5652
World           4770
New Age         2787
Name: count, dtype: int64


In [13]:
print(df3["majority_genre"].value_counts(normalize=True) * 100)

majority_genre
Rock          40.291136
Electronic    11.010893
Pop            9.883524
Jazz           6.246105
Rap            4.725974
RnB            4.703184
Metal          3.995998
Country        3.808340
Reggae         3.172727
Blues          2.893199
Latin          2.294618
Folk           2.270761
Punk           2.012598
World          1.698530
New Age        0.992412
Name: proportion, dtype: float64


In [14]:
df4.isna().sum()

track_id        0
mxm_track_id    0
word_counts     0
dtype: int64

**Top-250 tracks**

In [15]:
song_play_counts = df1.groupby("song_id")["play_count"].sum().reset_index()
song_play_counts.columns = ["song_id", "total_play_count"]

print("Уникальных song_id в play counts:", song_play_counts.shape[0])

Уникальных song_id в play counts: 384546


In [16]:
top_250 = song_play_counts.sort_values("total_play_count", ascending=False).head(250)

In [17]:
top_250 = top_250.merge(
    df2[["song_id", "artist", "title"]],
    on="song_id",
    how="left"
)

top_250_tracks = top_250[["artist", "title", "total_play_count"]].reset_index(drop=True)
top_250_tracks.columns = ["artist", "title", "play_count"]

top_250_tracks.index = top_250_tracks.index + 1
top_250_tracks.index.name = "rank"

print(top_250_tracks.shape)
print(top_250_tracks.head(10))

(250, 3)
                                                 artist  \
rank                                                      
1                                         Dwight Yoakam   
2                                                 Björk   
3                                         Kings Of Leon   
4                                              Harmonia   
5     Barry Tuckwell/Academy of St Martin-in-the-Fie...   
6                                Florence + The Machine   
7                                           OneRepublic   
8                                      Five Iron Frenzy   
9                                              Tub Ring   
10                                            Sam Cooke   

                                                  title  play_count  
rank                                                                 
1                                        You're The One      726885  
2                                                  Undo      648239  
3 

In [18]:
print("Пропуски в artist:", top_250_tracks["artist"].isna().sum())

Пропуски в artist: 0


**Top-100 tracks by genre**

In [19]:
rock_tracks = df3[df3["majority_genre"] == "Rock"][["track_id"]]

In [20]:
rock_songs = rock_tracks.merge(df2[["track_id", "song_id", "artist", "title"]], on="track_id", how="left")

In [21]:
rock_result = rock_songs.merge(song_play_counts, on="song_id", how="inner")

In [22]:
rock_result = rock_result.sort_values("total_play_count", ascending=False).head(100)
rock_top100 = rock_result[["artist", "title", "total_play_count"]].reset_index(drop=True)
rock_top100.columns = ["artist", "title", "play_count"]

rock_top100.index = rock_top100.index + 1
rock_top100.index.name = "rank"

In [23]:
print(rock_top100.shape)
print(rock_top100.head(10))

(100, 3)
             artist                               title  play_count
rank                                                               
1             Björk                                Undo      648239
2     Kings Of Leon                             Revelry      527893
3          Harmonia                       Sehr kosmisch      425463
4       OneRepublic                             Secrets      292642
5          Tub Ring                             Invalid      268353
6     Kings Of Leon                        Use Somebody      145725
7        The Crests                          16 Candles      129069
8          Coldplay                              Clocks      114362
9          Coldplay                              Yellow      109566
10         Paramore  The Only Exception (Album Version)      103653


**Collections**

1) Baseline

In [24]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

def get_word_count(word_counts_str, idx):
    for pair in word_counts_str.split(","):
        i, cnt = pair.split(":")
        if int(i) == idx:
            return int(cnt)
    return 0

def top50_by_keyword(word, threshold=5):
    stemmed_word = stemmer.stem(word)
    word_index = vocab.index(stemmed_word) + 1

    word_counts_col = df4["word_counts"].apply(lambda x: get_word_count(x, word_index))
    tracks = df4[["track_id"]].copy()
    tracks["word_count_temp"] = word_counts_col

    tracks = tracks[tracks["word_count_temp"] >= threshold]

    songs = tracks.merge(df2[["track_id", "song_id", "artist", "title"]], on="track_id", how="left")

    result = songs.merge(song_play_counts, on="song_id", how="inner")

    result = result.sort_values("total_play_count", ascending=False).head(50)
    result = result[["artist", "title", "total_play_count"]].reset_index(drop=True)
    result.columns = ["artist", "title", "play_count"]

    result.index = result.index + 1
    result.index.name = "rank"

    return result

In [25]:
love_top50 = top50_by_keyword("war")
print(love_top50.shape)
display(love_top50.head(10))

(50, 3)


,artist,title,play_count
rank,,,
1,Guns N' Roses,Civil War,22893
2,Slipknot,'Til We Die (Album Version),15254
3,Carnal Forge,Sacred Flame,11271
4,John Mayer,War Of My Life,8893
5,Portugal. The Man,People Say,5911
6,Iron Maiden,These Colours Don't Run,4811
7,Sick Puppies,War,4545
8,Red Hot Chili Peppers,The Righteous And The Wicked (Album Version),4466
9,Kanye West,The Glory,3934


2. Word2vec

In [26]:
import gensim.downloader as api
w2v_model = api.load("glove-wiki-gigaword-100")

In [27]:
def top50_by_keyword_w2v(word, topn_similar=10, threshold=5):
    similar = w2v_model.most_similar(word, topn=topn_similar)
    similar_words = [w for w, _ in similar] + [word]

    word_indices = []
    for w in similar_words:
        stemmed = stemmer.stem(w)
        if stemmed in vocab:
            idx = vocab.index(stemmed) + 1
            word_indices.append(idx)

    word_indices = list(set(word_indices))
    print(f"Найдено {len(word_indices)} индексов слов из {len(similar_words)} похожих слов")

    def get_total_count(word_counts_str, indices):
        total = 0
        for pair in word_counts_str.split(","):
            i, cnt = pair.split(":")
            if int(i) in indices:
                total += int(cnt)
        return total

    tracks = df4[["track_id"]].copy()
    tracks["word_count_temp"] = df4["word_counts"].apply(lambda x: get_total_count(x, word_indices))

    tracks = tracks[tracks["word_count_temp"] >= threshold]

    songs = tracks.merge(df2[["track_id", "song_id", "artist", "title"]], on="track_id", how="left")

    result = songs.merge(song_play_counts, on="song_id", how="inner")

    result = result.sort_values("total_play_count", ascending=False).head(50)
    result = result[["artist", "title", "total_play_count"]].reset_index(drop=True)
    result.columns = ["artist", "title", "play_count"]

    result.index = result.index + 1
    result.index.name = "rank"
    
    return result

In [28]:
love_top50_w2v = top50_by_keyword_w2v("war")
print(love_top50_w2v.shape)
display(love_top50_w2v.head(10))

Найдено 6 индексов слов из 11 похожих слов
(50, 3)


,artist,title,play_count
rank,,,
1,Michael Jackson,Thriller,26038
2,The Casualties,Fight For Your Life,24884
3,Rage Against The Machine,Killing In The Name,24485
4,Guns N' Roses,Civil War,22893
5,Amon Amarth,North Sea Storm (Live),17382
6,Hollywood Undead,Young,16920
7,Eagle-Eye Cherry,Save Tonight,16702
8,Dispatch,The General,15685
9,Slipknot,'Til We Die (Album Version),15254


3. Classification task

In [29]:
from scipy.sparse import lil_matrix

n_tracks = len(df4)
n_words = len(vocab)

X = lil_matrix((n_tracks, n_words))

for row_idx, word_counts_str in enumerate(df4["word_counts"]):
    for pair in word_counts_str.split(","):
        word_idx, cnt = pair.split(":")
        X[row_idx, int(word_idx) - 1] = int(cnt)

X = X.tocsr()
print(X.shape)

(210519, 5000)


In [30]:
from sklearn.linear_model import LogisticRegression

def top50_by_keyword_classifier(word, threshold=1):
    stemmed_word = stemmer.stem(word)
    word_index = vocab.index(stemmed_word)

    word_column = X[:, word_index].toarray().flatten()
    labels = (word_column >= threshold).astype(int)
    print("Положительных примеров:", labels.sum(), "из", len(labels))

    X_without_target = X.copy()
    X_without_target[:, word_index] = 0

    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_without_target, labels)

    probabilities = clf.predict_proba(X_without_target)[:, 1]

    tracks = df4[["track_id"]].copy()
    tracks["probability"] = probabilities

    songs = tracks.merge(df2[["track_id", "song_id", "artist", "title"]], on="track_id", how="left")
    result = songs.merge(song_play_counts, on="song_id", how="inner")

    result = result.sort_values("probability", ascending=False).head(200)
    result = result.sort_values("total_play_count", ascending=False).head(50)
    result = result[["artist", "title", "total_play_count"]].reset_index(drop=True)
    result.columns = ["artist", "title", "play_count"]

    result.index = result.index + 1
    result.index.name = "rank"
    
    return result

In [31]:
war_top50_clf = top50_by_keyword_classifier("war")
print(war_top50_clf.shape)
display(war_top50_clf.head(10))

Положительных примеров: 7972 из 210519


/Users/ggang/Projects/DS3_My_Spotify_ID_886382-1/.venv/lib/python3.11/site-packages/scipy/sparse/_index.py:216: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


(50, 3)


,artist,title,play_count
rank,,,
1,Guns N' Roses,Civil War,22893
2,Lupe Fiasco feat. Biship G and Nikki Jean,Little Weapon (feat. Bishop G and Nikki Jean) ...,6928
3,Lupe Fiasco,Streets On Fire (Explicit Album Version),5168
4,Manowar,The Warriors Prayer (LP Version),4155
5,Flobots,Stand Up,3944
6,Kanye West,The Glory,3934
7,Rage Against The Machine,Calm Like A Bomb,2455
8,Silkk The Shocker/Master P/C-Murder,We Won't Stop (feat. Master P & C-Murder),2036
9,AZ,Let's Toast,1953


**People similar to you listening**

In [43]:
from sklearn.preprocessing import LabelEncoder

user_encoder = LabelEncoder()
song_encoder = LabelEncoder()

df1["user_idx"] = user_encoder.fit_transform(df1["user_id"])
df1["song_idx"] = song_encoder.fit_transform(df1["song_id"])

n_users = df1["user_idx"].nunique()
n_songs = df1["song_idx"].nunique()
print("users:", n_users, "songs:", n_songs)

users: 1019318 songs: 384546


In [44]:
np.random.seed(42)
test_fraction = 0.2
min_interactions = 5

df1_shuffled = df1.sample(frac=1, random_state=42).reset_index(drop=True)
df1_shuffled["rank_in_user"] = df1_shuffled.groupby("user_idx").cumcount()
user_counts = df1_shuffled.groupby("user_idx")["user_idx"].transform("count")

is_test = (df1_shuffled["rank_in_user"] < (user_counts * test_fraction).astype(int)) & (user_counts >= min_interactions)

train_df = df1_shuffled[~is_test].reset_index(drop=True)
test_df = df1_shuffled[is_test].reset_index(drop=True)

print("train:", train_df.shape, "test:", test_df.shape)

train: (39085529, 7) test: (9288057, 7)


In [45]:
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

train_matrix = csr_matrix(
    (np.ones(len(train_df)), (train_df["user_idx"], train_df["song_idx"])),
    shape=(n_users, n_songs)
)

train_matrix_norm = normalize(train_matrix, axis=1)
print(train_matrix.shape)

(1019318, 384546)


In [46]:
def get_recommendations_song_idx(user_idx, top_n_similar=50, top_k=10):
    user_vector = train_matrix_norm[user_idx]
    similarities = user_vector.dot(train_matrix_norm.T).toarray().flatten()
    similarities[user_idx] = 0

    similar_user_indices = np.argsort(similarities)[::-1][:top_n_similar]
    similar_scores = similarities[similar_user_indices]

    similar_users_matrix = train_matrix[similar_user_indices]
    song_scores = similar_users_matrix.T.dot(similar_scores)
    song_scores = np.asarray(song_scores).flatten()

    already_listened = train_matrix[user_idx].toarray().flatten() > 0
    song_scores[already_listened] = -1

    top_song_indices = np.argsort(song_scores)[::-1][:top_k]
    return top_song_indices


def recommend_for_user(user_id, top_n_similar=50, top_k=10):
    user_idx = user_encoder.transform([user_id])[0]
    top_song_indices = get_recommendations_song_idx(user_idx, top_n_similar, top_k)
    top_song_ids = song_encoder.inverse_transform(top_song_indices)

    result = pd.DataFrame({"song_id": top_song_ids})
    result = result.merge(df2[["song_id", "artist", "title"]], on="song_id", how="left")
    result = result[["artist", "title"]].reset_index(drop=True)
    return result

In [47]:
sample_user_id = df1["user_id"].iloc[0]
recommendations = recommend_for_user(sample_user_id)
display(recommendations)

,artist,title
0,Takka Takka,Fever
1,Josh Rouse,Come Back [Light Therapy]
2,Shout Out Louds,Very Loud (Soundtrack Version)
3,Josh Rouse,My Love Has Gone
4,We Are Scientists,After Hours
5,Josh Rouse,Love Vibration
6,Paul Tiernan,How To Say Goodbye (Soundtrack Version)
7,Eric Hutchinson,Rock & Roll (Album Version)
8,Band Of Horses,Our Swords (Soundtrack Version)
9,Fleet Foxes,White Winter Hymnal


In [48]:
def precision_at_k(user_idx, k=10, top_n_similar=50):
    top_song_indices = get_recommendations_song_idx(user_idx, top_n_similar=top_n_similar, top_k=k)
    test_songs = set(test_df[test_df["user_idx"] == user_idx]["song_idx"])
    if len(test_songs) == 0:
        return None
    hits = len(set(top_song_indices) & test_songs)
    return hits / k

In [49]:
np.random.seed(42)
users_with_test = test_df["user_idx"].unique()
sample_users = np.random.choice(users_with_test, size=1000, replace=False)

precisions = [p for u in sample_users if (p := precision_at_k(u, k=10)) is not None]
avg_precision = np.mean(precisions)
print(f"Средний p@10 по {len(precisions)} пользователям: {avg_precision:.2%}")

Средний p@10 по 1000 пользователям: 12.59%


**People who listen to this track usually listen**

In [55]:
song_matrix = train_matrix.T
song_matrix_norm = normalize(song_matrix, axis=1)
print(song_matrix_norm.shape)

(384546, 1019318)


In [ ]:
def get_similar_tracks_song_idx(song_idx, top_k=10):
    song_vector = song_matrix_norm[song_idx]
    similarities = song_vector.dot(song_matrix_norm.T).toarray().flatten()
    similarities[song_idx] = 0

    top_song_indices = np.argsort(similarities)[::-1][:top_k]
    return top_song_indices


def recommend_similar_tracks(song_id, top_k=10):
    song_idx = song_encoder.transform([song_id])[0]
    top_song_indices = get_similar_tracks_song_idx(song_idx, top_k)
    top_song_ids = song_encoder.inverse_transform(top_song_indices)

    result = pd.DataFrame({"song_id": top_song_ids})
    result = result.merge(df2[["song_id", "artist", "title"]], on="song_id", how="left")
    result = result[["artist", "title"]].reset_index(drop=True)
    return result

In [57]:
sample_song_id = df1["song_id"].iloc[0]
similar_tracks = recommend_similar_tracks(sample_song_id)
display(similar_tracks)

,artist,title
0,Jack Johnson,Moonshine
1,Jack Johnson,Holes To Heaven
2,Jack Johnson / G. Love,Rainbow
3,Jack Johnson,Questions
4,Jack Johnson,Supposed To Be
5,Jack Johnson,Wrong Turn
6,Jack Johnson,People Watching
7,Jack Johnson / Matt Costa / Zach Gill / Dan Le...,Let It Be Sung
8,Jack Johnson / Matt Costa,Lullaby
9,Jack Johnson,They Do_ They Don't


In [ ]:
def get_recommendations_for_user_item_based(user_idx, top_k=10):
    train_songs = train_df[train_df["user_idx"] == user_idx]["song_idx"].values
    if len(train_songs) == 0:
        return None

    user_songs_matrix = song_matrix_norm[train_songs]
    similarities_sum = user_songs_matrix.dot(song_matrix_norm.T).toarray().sum(axis=0)

    similarities_sum[train_songs] = -1

    top_song_indices = np.argsort(similarities_sum)[::-1][:top_k]
    return top_song_indices

In [59]:
def precision_at_k_item_based(user_idx, k=10):
    top_song_indices = get_recommendations_for_user_item_based(user_idx, top_k=k)
    if top_song_indices is None:
        return None

    test_songs = set(test_df[test_df["user_idx"] == user_idx]["song_idx"])
    if len(test_songs) == 0:
        return None

    hits = len(set(top_song_indices) & test_songs)
    return hits / k

In [60]:
np.random.seed(42)
users_with_test = test_df["user_idx"].unique()
sample_users = np.random.choice(users_with_test, size=1000, replace=False)

precisions = [p for u in sample_users if (p := precision_at_k_item_based(u, k=10)) is not None]
avg_precision = np.mean(precisions)
print(f"Средний p@10 по {len(precisions)} пользователям: {avg_precision:.2%}")

Средний p@10 по 1000 пользователям: 12.90%
